<a href="https://colab.research.google.com/github/mjnhcanhh/DA_DeepLearning/blob/main/Train_Yolov12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1 — Cài đặt & kết nối Drive
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics -q

from ultralytics import YOLO
import os, shutil, yaml, torch, csv, time, threading
import numpy as np
from pathlib import Path

import ultralytics
print(f'✅ Ultralytics: {ultralytics.__version__}')
print('✅ Sẵn sàng!')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics: 8.4.41
✅ Sẵn sàng!


In [ ]:
# CELL 2 — CẤU HÌNH
ZIP_PATH     = '/content/drive/MyDrive/dataset_final.zip'
FINAL_PATH   = '/content/dataset_final'
PROJECT_PATH = '/content/drive/MyDrive/YOLOv12_Project'
RUN_NAME     = 'accident_v4'   # KHÔNG ĐỔI
TOTAL_EPOCHS = 100
NC           = 2
CLASS_NAMES  = ['accident', 'normal']

WEIGHTS_DIR  = f'{PROJECT_PATH}/{RUN_NAME}/weights'
RESULTS_PATH = f'{PROJECT_PATH}/{RUN_NAME}/results_total.csv'

print(f'📁 Dataset : {ZIP_PATH}')
print(f'📁 Project : {PROJECT_PATH}/{RUN_NAME}')
print(f'📊 Epochs  : {TOTAL_EPOCHS}')

📁 Dataset : /content/drive/MyDrive/dataset_final.zip
📁 Project : /content/drive/MyDrive/YOLOv12_Project/accident_v4
📊 Epochs  : 100


In [ ]:
# CELL 3 — Fix duplicate CSV + Tìm checkpoint
import pandas as pd

# Fix duplicate results_total.csv
if os.path.exists(RESULTS_PATH):
    df = pd.read_csv(RESULTS_PATH)
    before = len(df)
    df = df.drop_duplicates(subset=['epoch_total'], keep='first')
    df = df.sort_values('epoch_total').reset_index(drop=True)
    df.to_csv(RESULTS_PATH, index=False)
    print(f'✅ Fix duplicate: {before} → {len(df)} epochs')

# Tìm checkpoint
def find_best_checkpoint(weights_dir):
    if not os.path.exists(weights_dir):
        return None, 0
    candidates = []
    print(f'🔍 Kiểm tra: {weights_dir}')
    for f in sorted(os.listdir(weights_dir)):
        if not f.endswith('.pt'):
            continue
        path = os.path.join(weights_dir, f)
        try:
            ckpt  = torch.load(path, map_location='cpu', weights_only=False)
            epoch = int(ckpt.get('epoch', -1)) + 1
            candidates.append((epoch, f, path))
            print(f'  ✅ {f}: epoch {epoch}')
        except:
            print(f'  ❌ {f}: corrupt — bỏ qua')
    if not candidates:
        return None, 0
    candidates.sort(reverse=True)
    best_epoch, best_file, best_path = candidates[0]
    print(f'\n🎯 Checkpoint: {best_file} (epoch {best_epoch})')
    return best_path, best_epoch

CHECKPOINT_PATH, trained_epochs = find_best_checkpoint(WEIGHTS_DIR)
remaining = TOTAL_EPOCHS - trained_epochs

if CHECKPOINT_PATH:
    print(f'\n📊 Đã train: {trained_epochs}/{TOTAL_EPOCHS} epochs')
    print(f'⏳ Còn lại : {remaining} epochs')
else:
    print('🆕 Train mới từ đầu với YOLOv12n')

✅ Fix duplicate: 15 → 15 epochs
🔍 Kiểm tra: /content/drive/MyDrive/YOLOv12_Project/accident_v4/weights
  ✅ best.pt: epoch 44
  ✅ epoch0.pt: epoch 1
  ✅ epoch10.pt: epoch 11
  ✅ epoch15.pt: epoch 16
  ✅ epoch20.pt: epoch 21
  ✅ epoch25.pt: epoch 26
  ✅ epoch30.pt: epoch 31
  ✅ epoch35.pt: epoch 36
  ✅ epoch40.pt: epoch 41
  ✅ epoch5.pt: epoch 6
  ✅ last.pt: epoch 44

🎯 Checkpoint: last.pt (epoch 44)

📊 Đã train: 44/100 epochs
⏳ Còn lại : 56 epochs


In [ ]:
# CELL 4 — Giải nén dataset
import zipfile

if os.path.exists(FINAL_PATH) and len(os.listdir(FINAL_PATH)) > 0:
    print('✅ Dataset đã có sẵn:')
    for s in ['train', 'valid', 'test']:
        n = len(list((Path(FINAL_PATH) / s / 'images').glob('*')))
        print(f'   {s}: {n} ảnh')
else:
    print('📦 Đang giải nén...')
    os.makedirs('/content/tmp_extract', exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/tmp_extract')
    items = os.listdir('/content/tmp_extract')
    if len(items) == 1 and os.path.isdir(f'/content/tmp_extract/{items[0]}'):
        shutil.move(f'/content/tmp_extract/{items[0]}', FINAL_PATH)
    else:
        shutil.move('/content/tmp_extract', FINAL_PATH)
    print('✅ Giải nén xong!')
    for s in ['train', 'valid', 'test']:
        n = len(list((Path(FINAL_PATH) / s / 'images').glob('*')))
        print(f'   {s}: {n} ảnh')

📦 Đang giải nén...
✅ Giải nén xong!
   train: 31927 ảnh
   valid: 4003 ảnh
   test: 3984 ảnh


In [ ]:
# CELL 5 — Tạo data.yaml
yaml_path = os.path.join(FINAL_PATH, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump({
        'path' : FINAL_PATH,
        'train': 'train/images',
        'val'  : 'valid/images',
        'test' : 'test/images',
        'nc'   : NC,
        'names': CLASS_NAMES
    }, f, allow_unicode=True)
print(f'✅ data.yaml → {yaml_path}')

✅ data.yaml → /content/dataset_final/data.yaml


In [ ]:
# CELL 6 — Phân tích dataset
acc_total = nor_total = 0
for split in ['train', 'valid', 'test']:
    lbl_dir = Path(FINAL_PATH) / split / 'labels'
    acc = nor = 0
    for lbl in lbl_dir.glob('*.txt'):
        for l in lbl.read_text().strip().splitlines():
            cls = int(l.split()[0])
            if cls == 0: acc += 1
            else:        nor += 1
    acc_total += acc
    nor_total += nor
    imgs = len(list((Path(FINAL_PATH) / split / 'images').glob('*')))
    print(f'  {split:5s}: {imgs:5d} ảnh | accident={acc} | normal={nor}')

imbalance_ratio = max(acc_total, nor_total) / min(acc_total, nor_total)
cls_weights = None
if imbalance_ratio > 5:
    copy_paste, mixup, cls_weights = 0.3, 0.2, 0.5
elif imbalance_ratio > 2:
    copy_paste, mixup, cls_weights = 0.15, 0.15, 0.5
else:
    copy_paste, mixup = 0.1, 0.1
print(f'\n  Tỉ lệ mất cân bằng: {imbalance_ratio:.1f}x')

  train: 31927 ảnh | accident=49109 | normal=59942
  valid:  4003 ảnh | accident=6052 | normal=7560
  test :  3984 ảnh | accident=6136 | normal=7371

  Tỉ lệ mất cân bằng: 1.2x


In [ ]:
# CELL 7 — TRAIN YOLOV12

# Tải yolov12n.pt nếu chưa có
import urllib.request
if not CHECKPOINT_PATH and not os.path.exists('yolov12n.pt'):
    print('⬇️  Đang tải yolov12n.pt...')
    urllib.request.urlretrieve(
        'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov12n.pt',
        'yolov12n.pt'
    )
    print('✅ Tải xong!')

# TRAIN
if remaining <= 0:
    print(f'🎉 Đã đủ {TOTAL_EPOCHS} epochs!')
else:
    if CHECKPOINT_PATH:
        print(f'▶️  RESUME từ epoch {trained_epochs} → {TOTAL_EPOCHS}')
        model = YOLO(CHECKPOINT_PATH)
        model.train(
            resume  = True,
            epochs  = TOTAL_EPOCHS,
            device  = 0,
            workers = 8,
        )
    else:
        print('🆕 Train mới YOLOv12n từ đầu')
        model = YOLO('yolov12n.pt')
        model.train(
            data          = yaml_path,
            epochs        = TOTAL_EPOCHS,
            imgsz         = 640,
            batch         = 16,
            project       = PROJECT_PATH,
            name          = RUN_NAME,
            device        = 0,
            exist_ok      = True,
            save_period   = 5,
            workers       = 8,
            amp           = True,
            patience      = 100,
            mosaic        = 1.0,
            copy_paste    = copy_paste,
            mixup         = mixup,
            degrees       = 5.0,
            translate     = 0.1,
            scale         = 0.5,
            shear         = 1.0,
            perspective   = 0.0003,
            fliplr        = 0.5,
            flipud        = 0.0,
            hsv_h         = 0.015,
            hsv_s         = 0.7,
            hsv_v         = 0.4,
            dropout       = 0.1,
            weight_decay  = 0.0005,
            warmup_epochs = 3,
            close_mosaic  = 10,
            optimizer     = 'AdamW',
            lr0           = 0.001,
            lrf           = 0.01,
        )

    print('\n✅ Training hoàn tất!')

▶️  RESUME từ epoch 44 → 100
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_final/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=94, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/drive/MyDrive/YOLOv12_Project/accident_v4/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=accident_v4, nbs=

KeyboardInterrupt: 